# 00 Dataset Setup and Profile

## Project
**Airport Wildlife Strike and Aircraft Damage Analyzer**

This notebook checks the public FAA data file, identifies the strike-report worksheet, and records the dataset shape, fields, types, missing values, and label availability. It does not train a model.

## Dataset download
1. Open the [FAA Wildlife Strike Database](https://wildlife.faa.gov/home) and choose the public data download/export. The FAA also links its latest annual strike report from the [wildlife data page](https://www.faa.gov/airports/airport_safety/wildlife).
2. Save the downloaded ZIP or Excel export into `data/raw/`.
3. If you downloaded a ZIP, extract it into `data/raw/`. The notebook searches for an `.xlsx` workbook.
4. Run this notebook from top to bottom in VS Code or Jupyter. Keep the original download unchanged.

The FAA field guide describes fields such as incident date, airport, flight phase, aircraft type, wildlife species, damage, bird count, height, and estimated repair costs. The source records are reported strike incidents; they do not include all flights that did not have a strike.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
OUTPUT_DIR = PROJECT_DIR / "outputs"
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

excel_files = sorted(RAW_DIR.rglob("*.xlsx"))
if not excel_files:
    raise FileNotFoundError(
        f"No Excel file found under {RAW_DIR}. Download the public FAA export, "
        "extract it if needed, and put the .xlsx file in data/raw/."
    )

SOURCE_FILE = next((p for p in excel_files if p.name.lower() == "public.xlsx"), excel_files[0])
workbook = pd.ExcelFile(SOURCE_FILE)
print("Source workbook:", SOURCE_FILE.name)
print("Sheets:", workbook.sheet_names)

In [ ]:
# Prefer a worksheet whose name mentions strikes; otherwise inspect the first sheet.
candidate_sheets = [name for name in workbook.sheet_names if "strike" in name.lower()]
SHEET_NAME = candidate_sheets[0] if candidate_sheets else workbook.sheet_names[0]
raw = pd.read_excel(SOURCE_FILE, sheet_name=SHEET_NAME)

def normalize_column_name(name):
    name = str(name).strip().upper()
    name = re.sub(r"[^A-Z0-9]+", "_", name).strip("_")
    return name

raw.columns = [normalize_column_name(c) for c in raw.columns]
print("Selected sheet:", SHEET_NAME)
print("Raw shape:", raw.shape)
display(raw.head())

In [ ]:
column_profile = pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "missing_count": [int(raw[c].isna().sum()) for c in raw.columns],
    "missing_percent": [round(float(raw[c].isna().mean() * 100), 2) for c in raw.columns],
    "unique_non_null": [int(raw[c].nunique(dropna=True)) for c in raw.columns],
}).sort_values(["missing_percent", "column"], ascending=[False, True])

print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]:,}")
display(column_profile)
raw_summary = raw.describe(include="all").transpose()
display(raw_summary)

profile_path = OUTPUT_DIR / "dataset_column_profile.csv"
column_profile.to_csv(profile_path, index=False)
raw_summary.to_csv(OUTPUT_DIR / "dataset_descriptive_statistics.csv")
standardized_path = INTERIM_DIR / "strike_reports_standardized.csv"
raw.to_csv(standardized_path, index=False)
print("Saved standardized source copy:", standardized_path)
print("Damage fields found:", [c for c in ["DAMAGE", "INDICATED_DAMAGE"] if c in raw.columns])

## What to check before continuing
- Confirm the selected sheet contains one row per strike report and includes a damage field.
- Review missingness in `DAMAGE`, species, phase, time, aircraft, and bird-count fields.
- Write down the row and column counts shown above for the report.
- If the wrong worksheet was selected, set `SHEET_NAME` manually in Cell 3 and rerun.

### Output from this notebook
`data/interim/strike_reports_standardized.csv` and `outputs/dataset_column_profile.csv`.